# Heart Disease Prediction
---

## Initialize And Modify Data

In [ ]:
import pandas as pd

#Fetch data
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")


#Convert string data to int
train["Heart Disease"] = (train["Heart Disease"]=="Presence").astype(int)


#Add features
train["Hypertension"] = (train["BP"]>=120).astype(int)
test["Hypertension"] = (test["BP"]>=120).astype(int)

train = pd.get_dummies(train, columns=["Chest pain type", "Thallium"], drop_first=False, dtype=int)
test = pd.get_dummies(test, columns=["Chest pain type", "Thallium"], drop_first=False, dtype=int)

train["Old"] = (train["Age"]>55).astype(int)
test["Old"] = (test["Age"]>55).astype(int)

#train["ST_Slope_Interaction"] = train["Slope of ST"] * train["ST depression"]
#test["ST_Slope_Interaction"] = test["Slope of ST"] * test["ST depression"]

#train["HR_Angina"] = train["Max HR"] * train["Exercise angina"]
#test["HR_Angina"] = test["Max HR"] * test["Exercise angina"]

#train["Chol_Age_ratio"] = train["Cholesterol"] / (train["Age"])
#test["Chol_Age_ratio"] = test["Cholesterol"] / (test["Age"])



features = [
    'Chest pain type_4',
    'Sex',
    'Max HR',
    'Slope of ST',
    'Exercise angina',
    'Number of vessels fluro',
    'ST depression',
    'Chest pain type_3',
    'EKG results',
    'Old',
    'Hypertension',
    'Age',
    'Chest pain type_2',
    'Chest pain type_1',
    'Cholesterol',
    'BP',
    'FBS over 120',
    'Thallium_3',
    'Thallium_6',
    'Thallium_7'
 ]

train[features]

## Hyperparameter Tuning

Hyperparameter tuning for XGBoost and Light GBM models.

In [ ]:
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
    'n_estimators': trial.suggest_int('n_estimators', 100, 1500),  
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
    'max_depth': trial.suggest_int('max_depth', 3, 12),
    'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),  
    'gamma': trial.suggest_float('gamma', 0, 0.5),                     
    'device': 'cuda'
}
    model = XGBClassifier(**params)
    score = cross_val_score(model, train[features], train["Heart Disease"], cv=5, scoring='roc_auc').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print(study.best_params)
print("Best score:", study.best_value)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
import optuna

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    }
    model = LGBMClassifier(**params)
    score = cross_val_score(model, train[features], train["Heart Disease"], cv=5, scoring='roc_auc').mean()
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)
print(study.best_params)

## Train Model

In [ ]:
xgb_params = {
    'n_estimators': 576, 
    'learning_rate': 0.08412193755694605, 
    'max_depth': 3, 
    'subsample': 0.9918203954829311, 
    'colsample_bytree': 0.9055638012869013, 
    'min_child_weight': 4, 
    'gamma': 0.20965337793656907
}

lgb_params = {
    'n_estimators': 838, 
    'learning_rate': 0.1076, 
    'num_leaves': 153, 
    'max_depth': 3, 
    'min_child_samples': 87, 
    'subsample': 0.93, 
    'colsample_bytree': 0.536,
    'verbose': -1,
    'force_row_wise':True,
}

cat_params = {
    'iterations': 880, 
    'learning_rate': 0.11245020016843169, 
    'depth': 4, 'l2_leaf_reg': 10, 
    'random_strength': 0.6727445963744507, 
    'border_count': 92,
    'verbose': False
}

rf_params = {
    'n_estimators': 500,
    'max_depth': 7,         
    'min_samples_split': 2,    
    'min_samples_leaf': 1,     
    'max_features': 'sqrt',     
    'class_weight': 'balanced', 
    'random_state': 42
}

In [ ]:
import xgboost as xgb
import lightgbm as lgbm
import catboost as cat
from sklearn.ensemble import RandomForestClassifier as rf
from sklearn.model_selection import KFold
import numpy as np
from sklearn.linear_model import LogisticRegression

# Train models on full Dataset 

xgb_model = xgb.XGBClassifier(**xgb_params)
lgbm_model = lgbm.LGBMClassifier(**lgb_params)
cat_model = cat.CatBoostClassifier(**cat_params)

models = [xgb_model, lgbm_model, cat_model]

for model in models:
    model.fit(train[features], train["Heart Disease"])

# OOF 

model_ = [xgb.XGBClassifier(**xgb_params), lgbm.LGBMClassifier(**lgb_params), cat.CatBoostClassifier(**cat_params)]

oof_preds = np.zeros((len(train[features]), 3))  # 3 models
kf = KFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(kf.split(train[features])):
    X_fold_train, X_fold_val = train[features].iloc[train_idx], train[features].iloc[val_idx]
    y_fold_train, y_fold_val = train["Heart Disease"].iloc[train_idx], train["Heart Disease"].iloc[val_idx]

    for i, model in enumerate(model_):
        m = model
        m.fit(X_fold_train, y_fold_train)
        oof_preds[val_idx, i] = m.predict_proba(X_fold_val)[:, 1]

# Meta Model 

meta_model = LogisticRegression()
meta_model.fit(oof_preds, train["Heart Disease"])

#Final prediction
test_preds = np.column_stack([
    xgb_model.predict_proba(test[features])[:, 1],
    lgbm_model.predict_proba(test[features])[:, 1],
    cat_model.predict_proba(test[features])[:, 1]
])
final = meta_model.predict_proba(test_preds)[:, 1]

final

## Upload Prediction to CSV

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "Heart Disease": final    
})

submission.to_csv("data/submission.csv", index=False)